# GeoModelTest 台词配音（Qwen3-TTS）

在 Colab 上为游戏剧情生成日语配音。顺序：**1 准备 → 2 设计声音（试听挑选）→ 3 用选定的声音生成全部台词 → 4 下载 zip**。

- 运行时间：先在「运行时 → 更改运行时类型」选 GPU。L4 或 A100 最好；T4 也能跑（用 float16、不装 flash-attn），会慢一些。
- 模型：Qwen3-TTS 1.7B（Apache-2.0，可用于游戏）。VoiceDesign 用文字描述做出声音，Base 用选好的样本克隆，保证 100 多句都是同一个声音。
- 输入：仓库里的 `tools/voice/voice_lines.json`（`python3 tools/voice/export_voice_lines.py` 生成）。剧本改了就重新导出、重新跑第 3 步。
- 输出：`Voice/ja-JP/<textKey>.mp3` + `manifest.json`。解压后放进 Unity 工程的 `Assets/StreamingAssets/Voice/ja-JP/`。

## 1. 准备

In [ ]:
!pip install -q -U qwen-tts soundfile
import torch, subprocess
gpu = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none'
print('GPU:', gpu)
# Ampere 以后（L4 / A100）用 bf16 + flash-attn；T4 等用 fp16 + sdpa
MAJOR = torch.cuda.get_device_capability(0)[0] if torch.cuda.is_available() else 0
DTYPE = torch.bfloat16 if MAJOR >= 8 else torch.float16
ATTN = 'sdpa'
if MAJOR >= 8:
    r = subprocess.run('MAX_JOBS=4 pip install -q -U flash-attn --no-build-isolation', shell=True)
    ATTN = 'flash_attention_2' if r.returncode == 0 else 'sdpa'
print('dtype', DTYPE, 'attention', ATTN)

上传 `voice_lines.json`（在仓库 `tools/voice/` 里）。

In [ ]:
import json, os
from google.colab import files
if not os.path.exists('voice_lines.json'):
    files.upload()
LINES = json.load(open('voice_lines.json', encoding='utf-8'))
from collections import Counter
print(len(LINES), 'lines', Counter(l['speaker'] for l in LINES))

## 2. 设计声音（每个角色出 3 个候选，听完挑一个）

改 `VOICES` 里的描述可以调声音。主人公是性别模糊的中学一年级学生；旁白冷静清楚。试听句用游戏里的真实台词。

In [ ]:
import soundfile as sf
from IPython.display import Audio, display
from qwen_tts import Qwen3TTSModel

VOICES = {
    'kaede': ('20代後半の明るい女性研究者。好奇心いっぱいで、中学生にやさしく話しかける。はっきりした発音、少し速めでテンポがいい。',
              'わたしはカエデ。好きなものは、地層とおかし！'),
    'player': ('中学1年生。男の子とも女の子とも聞こえる、中性的で素直な声。おどろきや発見が自然に出る話し方。',
               '……ここ、どこ？　わかった、やってみる！'),
    'narration': ('落ち着いた大人のナレーター。ゆっくり、聞き取りやすく、場面の空気を伝える。',
                  '昼休みの教室。とつぜん、大きなゆれがきた！'),
}
CANDIDATES = 3
os.makedirs('design', exist_ok=True)
design = Qwen3TTSModel.from_pretrained('Qwen/Qwen3-TTS-12Hz-1.7B-VoiceDesign', device_map='cuda:0', dtype=DTYPE, attn_implementation=ATTN)
for role, (instruct, sample) in VOICES.items():
    for n in range(1, CANDIDATES + 1):
        torch.manual_seed(1000 * n + len(role))
        wavs, sr = design.generate_voice_design(text=sample, language='Japanese', instruct=instruct)
        path = f'design/{role}_{n}.wav'
        sf.write(path, wavs[0], sr)
        print(path); display(Audio(path))

在下面填每个角色选的编号（例如 `'kaede': 2`）。不满意就改上面的描述再跑一次。选好后释放 VoiceDesign 模型。

In [ ]:
PICK = {'kaede': 1, 'player': 1, 'narration': 1}
REFERENCE = {role: (f'design/{role}_{n}.wav', VOICES[role][1]) for role, n in PICK.items()}
del design; import gc; gc.collect(); torch.cuda.empty_cache()
print(REFERENCE)

## 3. 用选定的声音生成全部台词

Base 模型用选好的样本做「克隆提示」，每个角色只算一次，之后所有台词共用。默认读 `read`（术语已换成假名）；某句读错了，把它的 key 放进 `KANA_ONLY` 改用全假名 `read_kana` 再跑。已生成的文件会跳过，想重做某句就删掉对应 mp3。

In [ ]:
clone = Qwen3TTSModel.from_pretrained('Qwen/Qwen3-TTS-12Hz-1.7B-Base', device_map='cuda:0', dtype=DTYPE, attn_implementation=ATTN)
PROMPTS = {role: clone.create_voice_clone_prompt(ref_audio=wav, ref_text=text, x_vector_only_mode=False)
           for role, (wav, text) in REFERENCE.items()}
KANA_ONLY = set()      # 例: {'story.s2.l02'}
SKIP_SPEAKERS = set()  # 主人公不配音时设为 {'player'}
OUT = 'Voice/ja-JP'
os.makedirs(OUT, exist_ok=True)

def to_mp3(wav_path, mp3_path):
    # 单声道 44.1kHz 64kbps：每句约 20〜60KB，浏览器都能播放
    subprocess.run(['ffmpeg', '-y', '-loglevel', 'error', '-i', wav_path, '-af',
                    'silenceremove=start_periods=1:start_threshold=-50dB,areverse,silenceremove=start_periods=1:start_threshold=-50dB,areverse,loudnorm=I=-18:TP=-2',
                    '-ac', '1', '-ar', '44100', '-b:a', '64k', mp3_path], check=True)

todo = [l for l in LINES if l['speaker'] not in SKIP_SPEAKERS and not os.path.exists(f"{OUT}/{l['key']}.mp3")]
print(len(todo), 'to generate')
BATCH = 8
for role in PROMPTS:
    rows = [l for l in todo if l['speaker'] == role]
    for i in range(0, len(rows), BATCH):
        chunk = rows[i:i + BATCH]
        texts = [l['read_kana'] if l['key'] in KANA_ONLY else l['read'] for l in chunk]
        wavs, sr = clone.generate_voice_clone(text=texts, language=['Japanese'] * len(texts), voice_clone_prompt=PROMPTS[role])
        for line, wav in zip(chunk, wavs):
            sf.write('tmp.wav', wav, sr)
            to_mp3('tmp.wav', f"{OUT}/{line['key']}.mp3")
        print(role, min(i + BATCH, len(rows)), '/', len(rows))

抽几句试听，确认音色一致、术语读音对。

In [ ]:
import random
for l in random.sample([l for l in LINES if os.path.exists(f"{OUT}/{l['key']}.mp3")], 6):
    print(l['speaker'], l['key'], l['text']); display(Audio(f"{OUT}/{l['key']}.mp3"))

## 4. 写 manifest 并下载

In [ ]:
keys = sorted(f[:-4] for f in os.listdir(OUT) if f.endswith('.mp3'))
json.dump({'keys': keys}, open(f'{OUT}/manifest.json', 'w'), ensure_ascii=False)
size = sum(os.path.getsize(f'{OUT}/{f}') for f in os.listdir(OUT)) / 1e6
print(len(keys), 'files', round(size, 1), 'MB')
!cd /content && zip -qr voice_ja-JP.zip Voice design
files.download('voice_ja-JP.zip')

## 放进游戏
1. 解压，把 `Voice/ja-JP/` 整个放到 Unity 工程的 `Assets/StreamingAssets/Voice/ja-JP/`。
2. 打开游戏：有 `manifest.json` 就自动配音；没有就和以前一样安静。
3. 关配音：设置里把「キャラクターの声の音量」拖到 0，或者网址后加 `?voice=off`。
4. `design/` 里是选定声音的原始样本，留着以后补台词时用同一个声音。